In [1]:
%cd /content

!git clone https://github.com/Aishwarya-93/Deepfake-Detection-KYC.git

%cd Deepfake-Detection-KYC

/content
Cloning into 'Deepfake-Detection-KYC'...
remote: Enumerating objects: 204, done.
remote: Counting objects: 100% (204/204), done.
remote: Compressing objects: 100% (160/160), done.
remote: Total 204 (delta 103), reused 127 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (204/204), 3.64 MiB | 19.61 MiB/s, done.
Resolving deltas: 100% (103/103), done.
/content/Deepfake-Detection-KYC


In [2]:
!pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless
!pip install opencv-python==4.10.0.84

Found existing installation: opencv-python 5.0.0.93
Uninstalling opencv-python-5.0.0.93:
  Successfully uninstalled opencv-python-5.0.0.93
Found existing installation: opencv-contrib-python 4.13.0.92
Uninstalling opencv-contrib-python-4.13.0.92:
  Successfully uninstalled opencv-contrib-python-4.13.0.92
Found existing installation: opencv-python-headless 5.0.0.93
Uninstalling opencv-python-headless-5.0.0.93:
  Successfully uninstalled opencv-python-headless-5.0.0.93
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 MB 11.9 MB/s eta 0:00:00


In [3]:
import cv2

print(cv2.__version__)
print(hasattr(cv2, "CascadeClassifier"))

4.10.0
True


In [4]:
import os
from getpass import getpass

os.environ["KAGGLE_USERNAME"] = input("Kaggle Username: ")
os.environ["KAGGLE_KEY"] = getpass("Kaggle API Token: ")


Kaggle Username: aishwarya99990
Kaggle API Token: ··········


In [5]:
!kaggle datasets download -d xdxd003/ff-c23
!unzip -q ff-c23.zip -d data/raw/

Dataset URL: https://www.kaggle.com/datasets/xdxd003/ff-c23
License(s): other
100% 16.7G/16.7G [02:56<00:00, 101MB/s] 



In [6]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()

DATASET_ROOT = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "FaceForensics++_C23"
)

SPLIT_DIR = PROJECT_ROOT / "data" / "splits"

OUTPUT_ROOT = PROJECT_ROOT / "data" / "processed"

print("Project Root:", PROJECT_ROOT)
print("Dataset Root:", DATASET_ROOT)
print("Split Directory:", SPLIT_DIR)
print("Output Directory:", OUTPUT_ROOT)

print("\nDataset exists:", DATASET_ROOT.exists())
print("Splits exist:", SPLIT_DIR.exists())

Project Root: /content/Deepfake-Detection-KYC
Dataset Root: /content/Deepfake-Detection-KYC/data/raw/FaceForensics++_C23
Split Directory: /content/Deepfake-Detection-KYC/data/splits
Output Directory: /content/Deepfake-Detection-KYC/data/processed

Dataset exists: True
Splits exist: True


In [7]:
train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df = pd.read_csv(SPLIT_DIR / "validation.csv")
test_df = pd.read_csv(SPLIT_DIR / "test.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (4899, 5)
Validation: (1050, 5)
Test: (1051, 5)


In [8]:
print("Train labels:")
print(train_df["Label"].value_counts())

print("\nValidation labels:")
print(val_df["Label"].value_counts())

print("\nTest labels:")
print(test_df["Label"].value_counts())

Train labels:
Label
FAKE    4199
REAL     700
Name: count, dtype: int64

Validation labels:
Label
FAKE    900
REAL    150
Name: count, dtype: int64

Test labels:
Label
FAKE    901
REAL    150
Name: count, dtype: int64


In [9]:
sample_path = train_df.iloc[0]["File Path"]

video_path = DATASET_ROOT / sample_path

print("CSV path:")
print(sample_path)

print("\nActual path:")
print(video_path)

print("\nExists:")
print(video_path.exists())

CSV path:
Deepfakes/812_821.mp4

Actual path:
/content/Deepfake-Detection-KYC/data/raw/FaceForensics++_C23/Deepfakes/812_821.mp4

Exists:
True


In [10]:
from src.preprocessing import frame_extractor
from src.preprocessing import face_detector
from src.preprocessing import image_preprocessing

print("Modules imported successfully.")

Modules imported successfully.


In [ ]:
import shutil

# Clear and recreate output folders so stale files from a previous
# (buggy, filename-colliding) run cannot survive into the new dataset.
for split in ["train", "val", "test"]:
    for label in ["real", "fake"]:
        output_dir = OUTPUT_ROOT / split / label

        if output_dir.exists():
            shutil.rmtree(output_dir)

        output_dir.mkdir(
            parents=True,
            exist_ok=True
        )

print("Output folders cleared and recreated.")

Process the full identity-disjoint split exactly as written to `data/splits/{train,validation,test}.csv` (train=4200, validation=900, test=900 videos) — no subsetting.

In [ ]:
import re

import cv2


def build_source_id(video_path, dataset_root):
    """
    Build a deterministic, globally-unique identifier for a source video
    from its path relative to DATASET_ROOT.

    Deepfakes/500_592.mp4       -> "Deepfakes_500_592"
    FaceSwap/500_592.mp4        -> "FaceSwap_500_592"
    Face2Face/500_592.mp4       -> "Face2Face_500_592"

    Using the full relative path (rather than just video_path.stem)
    keeps identifiers unique across manipulation-method folders that
    reuse the same underlying video stem, and across any nested
    subfolders (e.g. DeepFakeDetection).
    """

    relative_path = video_path.relative_to(dataset_root).with_suffix("")

    sanitized_parts = [
        re.sub(r"[^A-Za-z0-9_-]", "_", part)
        for part in relative_path.parts
    ]

    return "_".join(sanitized_parts)


def process_dataset_split(
    df,
    split_name,
    frame_interval=10,
    max_frames_per_video=5
):

    detector = face_detector.load_face_detector()

    output_base = OUTPUT_ROOT / split_name

    # Tracks every output path written during this call, mapped to the
    # source video that wrote it, so we can detect collisions instead
    # of silently overwriting files.
    written_sources = {}

    total_videos = 0
    total_frames = 0
    total_faces = 0

    for _, row in df.iterrows():

        relative_path = Path(row["File Path"])

        video_path = DATASET_ROOT / relative_path

        label = row["Label"].lower()

        if label == "real":
            output_dir = output_base / "real"
        else:
            output_dir = output_base / "fake"

        output_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        if not video_path.exists():
            print(
                f"Missing video: {video_path}"
            )
            continue

        cap = cv2.VideoCapture(
            str(video_path)
        )

        if not cap.isOpened():
            print(
                f"Could not open: {video_path}"
            )
            continue

        frame_count = int(
            cap.get(cv2.CAP_PROP_FRAME_COUNT)
        )

        source_id = build_source_id(video_path, DATASET_ROOT)

        frame_index = 0
        frames_used = 0

        while True:

            success, frame = cap.read()

            if not success:
                break

            if frame_index % frame_interval == 0:

                total_frames += 1

                faces = face_detector.detect_faces(
                    frame,
                    detector
                )

                cropped_faces = (
                    face_detector.crop_faces(
                        frame,
                        faces
                    )
                )

                for face_index, face in enumerate(
                    cropped_faces
                ):

                    processed_face = (
                        image_preprocessing.resize_image(
                            face,
                            (224, 224)
                        )
                    )

                    output_file = (
                        output_dir
                        / f"{source_id}_"
                          f"frame{frame_index}_"
                          f"face{face_index}.jpg"
                    )

                    if output_file.exists():
                        existing_source = written_sources.get(output_file)

                        if existing_source != str(video_path):
                            raise RuntimeError(
                                f"Filename collision detected: "
                                f"{output_file} already exists "
                                f"(written by "
                                f"{existing_source or 'an unknown prior run, i.e. a stale file on disk'}"
                                f"), but is now requested by {video_path}. "
                                "Refusing to silently overwrite. If this "
                                "is left over from a previous run, clear "
                                "data/processed before regenerating."
                            )

                    cv2.imwrite(
                        str(output_file),
                        processed_face
                    )

                    written_sources[output_file] = str(video_path)

                    total_faces += 1

                frames_used += 1

                if frames_used >= max_frames_per_video:
                    break

            frame_index += 1

        cap.release()

        total_videos += 1

        if total_videos % 10 == 0:
            print(
                f"{split_name}: "
                f"{total_videos}/{len(df)} videos processed"
            )

    return {
        "videos_processed": total_videos,
        "frames_processed": total_frames,
        "faces_saved": total_faces
    }

In [ ]:
simulated_paths = [
    "Deepfakes/500_592.mp4",
    "FaceSwap/500_592.mp4",
    "Face2Face/500_592.mp4",
    "FaceShifter/500_592.mp4",
    "NeuralTextures/500_592.mp4",
]

source_ids = [
    build_source_id(DATASET_ROOT / p, DATASET_ROOT)
    for p in simulated_paths
]

for p, sid in zip(simulated_paths, source_ids):
    print(f"{p:30s} -> {sid}")

assert len(source_ids) == len(set(source_ids)), (
    "Collision detected in synthetic filename check!"
)

print("\nAll synthetic source IDs are unique.")

In [ ]:
train_results = process_dataset_split(
    train_df,
    "train",
    frame_interval=10,
    max_frames_per_video=5
)

print(train_results)

In [ ]:
val_results = process_dataset_split(
    val_df,
    "val",
    frame_interval=10,
    max_frames_per_video=5
)

print(val_results)

In [ ]:
test_results = process_dataset_split(
    test_df,
    "test",
    frame_interval=10,
    max_frames_per_video=5
)

print(test_results)

In [19]:
for split in ["train", "val", "test"]:

    real_count = len(
        list(
            (OUTPUT_ROOT / split / "real").glob("*.jpg")
        )
    )

    fake_count = len(
        list(
            (OUTPUT_ROOT / split / "fake").glob("*.jpg")
        )
    )

    print(
        f"{split}: "
        f"REAL={real_count}, "
        f"FAKE={fake_count}"
    )

train: REAL=346, FAKE=390
val: REAL=74, FAKE=64
test: REAL=64, FAKE=106


In [20]:
from pathlib import Path

ROOT = Path("data/processed")

for split in ["train", "val", "test"]:
    for label in ["real", "fake"]:
        folder = ROOT / split / label

        count = len(list(folder.glob("*.jpg")))

        print(f"{split}/{label}: {count} images")

train/real: 346 images
train/fake: 390 images
val/real: 74 images
val/fake: 64 images
test/real: 64 images
test/fake: 106 images
